In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

import logging
logging.getLogger('matplotlib.font_manager').setLevel(level=logging.CRITICAL)

In [ ]:
import itertools

In [ ]:
from matplotlib import pyplot as plt
from tqdm import tqdm
import itertools
import os
import numpy as np
import scipy.sparse
import pandas as pd
import re
import tifffile
import xmltodict
import importlib.util
import sys
import scanpy as sc
sc.set_figure_params(dpi=80)

import math
from tifffile.tifffile import TiffFile
import xmltodict
from scipy.stats import linregress
import shutil
import scipy as sp

import importlib.util
import sys


In [ ]:
import cuml
import cupy
import cudf
import cugraph
import cuspatial
import cupyx

In [ ]:

def lazy_import(module_name, path_to_file):
    spec = importlib.util.spec_from_file_location(module_name,path_to_file)
    foo = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = foo
    spec.loader.exec_module(foo)
    return foo

utils_dir = "../../utils"
general_utils =  lazy_import("general_utils",os.path.join(utils_dir, "general_utils.py"))
preprocessing_utils = lazy_import("preprocessing_utils",os.path.join(utils_dir, "preprocessing_utils.py"))

from general_utils import ismember, grep, grep_exclude
from preprocessing_utils import kneepoint


In [ ]:
def divide_csr_matrix_gpu(mat, size_factor):
    """Divide csr matrix using gpu row-wise

    Params:
        - mat: np.array
        - size_factor: np.array
    Returns:
        np.array normalized matrix
    """
    mat = cupyx.scipy.sparse.csr_matrix(mat.astype(np.float32))
    c = cupyx.scipy.sparse.diags(1 / size_factor)
    result = c @ mat
    return result.get()


def neighbors_gpu(mat, k=30):
    """Compute knn on a matrix using gpu

    Params:
        - mat: np.array
        - k: number of nearest neighbors
    Returns:
        tuple: distances and indices of knn (array format)
    """
    X_mat = cupy.array(mat)
    model = cuml.neighbors.NearestNeighbors(n_neighbors=k)
    model.fit(X_mat)
    distances, indices = model.kneighbors(X_mat)
    return (distances, indices)


def get_cugraph_from_csr_matrix(mat):
    """Get cugraph from csr_matrix

    Params:
        - mat: scipy.csr_matrix
    Returns:
        cugraph representation of matrix
    """
    sources, targets = mat.nonzero()
    edges = cudf.concat(
        [
            cudf.Series(sources.astype(np.int64), name="source"),  # sources
            cudf.Series(targets.astype(np.int64), name="destination"),  # targets
        ],
        axis=1,
    )
    G = cugraph.from_cudf_edgelist(edges)
    return G


def get_cugraph_from_array(mat):
    """Get cugraph from array

    Params:
        - mat: list of lists in np.array format, each row in the array corresponds to the source node, values correspond to indexes the connected nodes. It assumes that the row indexes are in the same space as the values in the array.

    Returns
    -------
        cugraph representation of matrix
    """
    targets = np.ravel(mat)
    n_neighbors_per_cell = [len(x) for x in mat]
    sources = np.ravel([[i] * k for i, k in zip(range(mat.shape[0]), n_neighbors_per_cell)])
    edges = cudf.concat(
        [
            cudf.Series(sources.astype(np.int64), name="source"),  # sources
            cudf.Series(targets.astype(np.int64), name="destination"),  # targets
        ],
        axis=1,
    )
    G = cugraph.from_cudf_edgelist(edges)
    return (G, edges)


def array_to_csr_matrix(mat):
    """Get csr_matrix from list of list graph representation

    Params:
        - mat: list of lists in np.array format, each row in the array corresponds to the source node, values correspond to indexes the connected nodes. It assumes that the row indexes are in the same space as the values in the array.

    Returns
    -------
        csr_matrix representation of graph
    """
    sources, targets = array_to_edges(mat)
    result = scipy.sparse.csr_matrix(([1] * len(sources), (sources, targets)), shape=[mat.shape[0]] * 2)
    return result

def array_to_edges(mat):
    """Get csr_matrix from list of list graph representation

    Params:
        - mat: list of lists in np.array format, each row in the array corresponds to the source node, values correspond to indexes the connected nodes. It assumes that the row indexes are in the same space as the values in the array.

    Returns
    -------
        sources and targets of the graph
    """
    targets = np.ravel(mat)
    n_neighbors_per_cell = [len(x) for x in mat]
    sources = np.ravel([[i] * k for i, k in zip(range(mat.shape[0]), n_neighbors_per_cell)])
    return sources, targets

def phenograph_gpu(
    G,
    min_size=-1,
    resolution=1,
):
    print("   Building Jaccard matrix")
    # Build jaccard-weighted graph in GPU
    jaccard_edges = cugraph.jaccard(G, edges[['source', 'destination']])
    G = cugraph.from_cudf_edgelist(jaccard_edges, *jaccard_edges.columns)
    
    print("   Finding clusters using louvain")
    # Cluster jaccard-weighted graph
    result, score = cugraph.louvain(G, **kwargs)
    
    # Sort clusters by size
    sizes = result['partition'].value_counts()
    sizes.loc[:] = cupy.where(sizes > min_size, cupy.arange(len(sizes)), -1)
    result['partition'] = result['partition'].map(sizes)
    
    # Sort by vertex (e.g. cell)
    clusters = result.sort_values('vertex')['partition'].values.get()
    return clusters

def leiden_gpu(G, resolution=1.0):
    print("   Getting Leiden clusters")
    result, score = cugraph.leiden(G, max_iter = 100, resolution = resolution, random_state = 777)
    # Sort by vertex (e.g. cell)
    clusters = result.sort_values('vertex')['partition'].values.get()
    return(clusters)

def normalize_pca(
    adata,
    log = True,
    excluded_genes = [],
    min_count_threshold_default = 25,
    min_count_criteria = 'fixed'
):
    if len(excluded_genes) > 0:
        adata.var['excluded'] = excluded_genes
    else:
        adata.var['excluded'] = [False] * adata.n_vars
    # Exclude cells that have less than 2 counts for non-excluded genes
    adata = adata[adata.layers['raw'][:,np.logical_not(adata.var['excluded'])].sum(axis=1) > 1,:].copy()
    
    adata.X = adata.layers['raw'].copy()
    size_factor = np.ravel(adata.X[:,np.logical_not(adata.var['excluded'])].sum(axis=1))
    adata.obs['size_factor'] = size_factor.copy()
    
    # QC metrics
    sc.pp.calculate_qc_metrics(
        adata,
        inplace=True,
        log1p=False,
        layer="raw"
    )
    obs_cols = ["total_counts", "n_genes_by_counts"]
    var_cols = ["total_counts", "n_cells_by_counts", "mean_counts"]
    adata.obs[
        [f"log10_{c}" for c in obs_cols]
    ] = np.log10(adata.obs[obs_cols]+1) # pseudocount 1
    adata.var[
        [f"log10_{c}" for c in var_cols]
    ] = np.log10(adata.var[var_cols]+1) # pseudocount 1

    original_n = adata.n_obs
    # Filter low library-size cells
    x = np.arange(adata.shape[0])
    counts_by_cell = adata.X[:,np.logical_not(adata.var['excluded'])].sum(1).A.T[0]
    if(min_count_criteria == 'fixed'):
        min_count_threshold = min_count_threshold_default
    else:
        y = np.sort(counts_by_cell)[::-1]
        idx = kneepoint(np.log10(y), x)
        min_count_threshold = max(25, y[idx])
    adata = adata[counts_by_cell >= min_count_threshold,:].copy()
    final_n = adata.n_obs
    print("   Excluded %d cells with less than %d counts" % (original_n - final_n, min_count_threshold))

    # Median library-size normalization
    print("   Normalizing data")
    size_factor = adata.obs['size_factor'].to_numpy()
    adata.X = divide_csr_matrix_gpu(adata.X, size_factor) * np.median(size_factor)
    adata.X = adata.X.astype(np.float32)
    
    # Log transform if indicated for embedding computation
    if(log):
        print("   Log transform with pseudocount 1")
        # Log-transformation (natural log, pseudocount of 1)
        sc.pp.log1p(adata)

    # Run PCA using GPU
    print("   Running PCA")
    counts_sparse_gpu = cupyx.scipy.sparse.csr_matrix(adata.X[:,np.logical_not(adata.var['excluded'])])
    model = cuml.PCA(n_components=adata.shape[1]-1)
    model.fit(X=counts_sparse_gpu)
    X_pca = model.transform(counts_sparse_gpu)
    
    adata.obsm['X_pca_max'] = X_pca.get().astype(np.float32)
    adata.uns['pca_explained_var'] = model.explained_variance_ratio_.get()
    
    # Log transform for downstream computation
    if(not log):
        # Log-transformation (natural log, pseudocount of 1)
        sc.pp.log1p(adata)
    
    return(adata)

def umap(
    X,
    n_neighbors = 30,
    min_dist = 0.05
):
    model = cuml.UMAP(
        min_dist=min_dist,
        spread=1,
        learning_rate=0.1,
        n_epochs=20000,
        n_neighbors = n_neighbors,
        verbose=True
    )
    X = cupy.array(X)
    model.fit(X=X)
    X_umap = model.transform(X)    
    return(X_umap.get())

In [ ]:
tenx_data_dir = "../xenium_rawdata/"
output_path = '../xenium_preprocessing_outputs'

knn_output_path = os.path.join(output_path, 'knn')
pca_output_path = os.path.join(output_path, 'pca')
umap_output_path = os.path.join(output_path, 'umap')
adata_base_output_path = os.path.join(output_path, 'adata_base')
obs_output_path = os.path.join(output_path, 'obs')
input_path = os.path.join(output_path, 'adata_single_sample')

os.makedirs(knn_output_path, exist_ok=True)
os.makedirs(pca_output_path, exist_ok=True)
os.makedirs(umap_output_path, exist_ok=True)
os.makedirs(adata_base_output_path, exist_ok=True)
os.makedirs(obs_output_path, exist_ok=True)

In [ ]:
def get_compiled_adata(selected_tier, selected_subset, use_log, explained_var, umap_n_neighbors, cluster_n_neighbors, selected_layer = 'raw_in_nucleus', umap_min_dist = 0.5):
    dataset_prefix = "adata_" + selected_tier + "_SUBSET_%s_LAYER_%s_LOG_%s" % (selected_subset, selected_layer, use_log)
    file_prefix = "%s_EXPLAINEDVAR_%d_KNN_%s" % (dataset_prefix, int(explained_var * 100), "%d")
    
    adata_base_file = os.path.join(adata_base_output_path, dataset_prefix + ".h5ad")
    obs_file = os.path.join(obs_output_path,file_prefix % (cluster_n_neighbors) + "__obs.h5ad" )
    umap_file = os.path.join(umap_output_path,file_prefix % (umap_n_neighbors) + "_MINDIST_%d__umap.csv.gz" % (int(umap_min_dist * 100)))

    print("Reading counts file")
    adata = sc.read_h5ad(adata_base_file)
    print("Reading obs file")
    obs = sc.read_h5ad(obs_file)
    print("Reading umap file")
    umap = pd.read_csv(umap_file, index_col = 0)
    
    adata.obsm['X_umap'] = umap.to_numpy()
    for my_obs in obs.obs.columns:
        adata.obs[my_obs] = obs.obs[my_obs].copy()
    return(adata)

selected_tier = 'tier0'
use_log = False
explained_var = 0.75
umap_n_neighbors = 10
cluster_n_neighbors = 30
selected_layer = 'raw_in_nucleus'
umap_min_dist = 0.05

adata_original = get_compiled_adata(selected_tier = selected_tier, selected_subset = "all", use_log = use_log, explained_var = explained_var, umap_n_neighbors = umap_n_neighbors, cluster_n_neighbors = cluster_n_neighbors, selected_layer = selected_layer, umap_min_dist = umap_min_dist)


Reading counts file


In [ ]:
np.unique(adata_original.obs['cell_type_0'])

In [ ]:
subset_dictionary = {
    'Epi': ['Epithelial'], 
    'TME': [
        'Adipose', 'Endothelial', 'Fibroblast', 'Glial',
        'Immune_B', 'Immune_GrMDC', 'Immune_Myeloid', 'Immune_Plasma',
        'Immune_TNKILC', 'MuralCells', 'Endothelial_Vascular', 'Endothelial_Lymphatic',
        'GialNerves', 'Mesothelial', 'MuralCells'
    ],
    'Immune': ['Immune_B', 'Immune_GrMDC', 'Immune_Myeloid', 'Immune_Plasma', 'Immune_TNKILC'],
    'Fibroblast': ['Fibroblast'],
    'Mural': ['MuralCells'],
    'Endothelial': ['Endothelial_Vascular', 'Endothelial_Lymphatic', 'Endothelial'],
    'Vessels': ['MuralCells', 'Endothelial'],
    'Stroma': ['Fibroblast', 'Glial', 'Adipose'],
    'Myeloid': ['Immune_GrMDC', 'Immune_Myeloid'],
    'Lymphoid': ['Immune_TNKILC', 'Immune_B', 'Immune_Plasma']
}

file_database = pd.read_csv("../support/xenium_file_manifest_tier_classification.csv", sep = ",", header = 0)
possible_tiers = ['tier0', 'tier3']
for my_tier in possible_tiers:
    print(np.unique(file_database['condition'][file_database[my_tier] == 1].to_numpy()))

['K2_MRTX1133' 'K2_ctrl' 'K2_shp53' 'K2_vehicle' 'K2d1_ctrl' 'K2d1_shp53'
 'K3_ctrl' 'K3_shp53' 'K4_ctrl']
['K2_ctrl' 'K2_vehicle' 'K2d1_ctrl' 'K3_ctrl']
['K2_ctrl' 'K2_shp53' 'K2_vehicle' 'K2d1_ctrl' 'K2d1_shp53' 'K3_ctrl'
 'K3_shp53']
['K2_MRTX1133' 'K2_vehicle']


In [ ]:
selected_layer = "raw_in_nucleus"
possible_cluster_k = [30]
possible_umap_k = [10]
possible_explained_var = [0.75]
possible_log = [False]
possible_tiers = ['tier0', 'tier3']
possible_min_dist = [0.5, 0.1, 0.05]
possible_subsets = ["Epi", "TME", "Myeloid", "Lymphoid", "Fibroblast", "Mural", "Endothelial"]

First, I compile adata objects of different tiers and compute the full PCA matrix. This precomputation will be used for calculating neighborhood graphs, embeddings and clusters with different n_neighbors and explained_variance parameters

In [ ]:
%%time
loop_generator = itertools.product(possible_tiers, possible_subsets, possible_log)
for selected_tier, selected_subset, use_log in loop_generator:
    file_prefix = "adata_" + selected_tier + "_SUBSET_" + selected_subset + "_LAYER_" + selected_layer + "_LOG_" + str(use_log)
    adata_output_file = os.path.join(adata_base_output_path, file_prefix + ".h5ad")
    if not os.path.exists(adata_output_file):
        print("Processing " + adata_output_file)
        # Filter original adata object
        cell_type_filter = ismember(adata_original.obs['cell_type_0'], subset_dictionary[selected_subset])[1]
        selected_prefix = file_database['prefix'][file_database[selected_tier] == 1].to_numpy()
        tier_filter = ismember(adata_original.obs['prefix'], selected_prefix)[1]
        valid_cells = np.logical_and(cell_type_filter, tier_filter)
        adata = adata_original[valid_cells,:].copy()
        
        # Filter genes
        frac_expressed = np.ravel((adata.X > 0).sum(axis=0)) / adata.n_obs
        adata.var['excluded'] = np.logical_or(adata.var['excluded'], frac_expressed <= 0.01)
        
        # Run PCA
        adata = normalize_pca(adata, log=use_log, excluded_genes = list(adata.var['excluded']))
        
        # Write PCA output
        adata_pca = sc.AnnData(adata.obsm['X_pca_max'].copy())
        adata_pca.obs_names = adata.obs_names
        adata_pca.var['pca_explained_var'] = adata.uns['pca_explained_var'].copy()
        sc.write(os.path.join(pca_output_path, file_prefix + "__pca.h5ad"), adata= adata_pca)
        
        # Write light adata
        del adata.obsm['X_pca_max']
        del adata.uns['pca_explained_var']
        del adata.obsm['X_umap']
        sc.write(adata_output_file, adata=adata)
    
        # Filter PCA according to possible explained variances
        for explained_var in possible_explained_var:
            output_pca_prefix = file_prefix + "_EXPLAINEDVAR_%d" % (int(explained_var * 100))
            output_pca_file = os.path.join(pca_output_path, output_pca_prefix + '__pca.h5ad')
            total_var = explained_var
            n_pcs = np.argmin(abs(adata_pca.var['pca_explained_var'].cumsum() - total_var))
            adata_result = sc.AnnData(adata_pca.X[:, :n_pcs])
            adata_result.obs_names = adata_pca.obs_names
            adata_result.uns['pca_explained_var'] = adata_pca.var['pca_explained_var'][:n_pcs].to_numpy()
            print("   Kept %d PCs that explain %.2f variance" % (n_pcs, total_var))
            sc.write(output_pca_file, adata=adata_result)

CPU times: user 350 µs, sys: 16 µs, total: 366 µs
Wall time: 1.77 ms


Now, I use the precompiled adata objects to process neighborhood graphs

In [ ]:
loop_generator = itertools.product(possible_tiers, possible_subsets, possible_log, possible_explained_var, possible_cluster_k)

for selected_tier, selected_subset, use_log, explained_var, n_neighbors in loop_generator:
    dataset_prefix = "adata_" + selected_tier + "_SUBSET_" + selected_subset + "_LAYER_" + selected_layer + "_LOG_" + str(use_log) + "_EXPLAINEDVAR_%d" % int(explained_var * 100)
    file_prefix = dataset_prefix + "_KNN_%d" % (n_neighbors)
    output_file = os.path.join(knn_output_path, file_prefix + '__knn.h5ad')
    if not os.path.exists(output_file):
        pca_file = os.path.join(pca_output_path, dataset_prefix + "__pca.h5ad")
        print("Reading " + pca_file)
        adata_pca = sc.read_h5ad(pca_file)
        distances, indices = neighbors_gpu(adata_pca.X, k=n_neighbors)
        print("Writing outputs")
        adata_knn = sc.AnnData(scipy.sparse.csr_matrix([[0] * n_neighbors] * adata_pca.n_obs))
        adata_knn.obsm['indices'] = indices
        adata_knn.obsm['distances'] = distances
        adata_knn.obs_names = adata_pca.obs_names.copy()
        sc.write(output_file, adata=adata_knn)


Here, I run UMAP on samples

In [ ]:
loop_generator = itertools.product(possible_tiers, possible_subsets, possible_log, possible_explained_var, possible_umap_k, possible_min_dist)

for selected_tier, selected_subset, use_log, explained_var, n_neighbors, min_dist in loop_generator:
    dataset_prefix = "adata_" + selected_tier + "_SUBSET_" + selected_subset + "_LAYER_" + selected_layer + "_LOG_" + str(use_log) + "_EXPLAINEDVAR_%d" % int(explained_var * 100)
    file_prefix = dataset_prefix + "_KNN_%d_MINDIST_%d" % (n_neighbors, int(min_dist * 100))
    output_file = os.path.join(umap_output_path, file_prefix + "__umap.csv.gz")
    if os.path.exists(output_file):
        continue
    pca_file = os.path.join(pca_output_path, dataset_prefix + "__pca.h5ad")
    print("Reading " + pca_file)
    adata_pca = sc.read_h5ad(pca_file)
    print("Computing UMAP")
    umap_result = umap(adata_pca.X, n_neighbors = n_neighbors, min_dist = min_dist)
    result = pd.DataFrame(umap_result)
    result.index = adata_pca.obs_names
    print("Writing outputs")
    result.to_csv(output_file)


Reading /data/lowe/reyesj3/pdac_loh/Xenium/preprocessing_outputs_3/pca/adata_tier0_SUBSET_TME_LAYER_raw_in_nucleus_LOG_False_EXPLAINEDVAR_75__pca.h5ad
Computing UMAP
[D] [11:42:16.325767] /__w/cuml/cuml/cpp/src/umap/runner.cuh:108 n_neighbors=10
[D] [11:42:16.326792] /__w/cuml/cuml/cpp/src/umap/runner.cuh:130 Calling knn graph run
[D] [12:58:44.297525] /__w/cuml/cuml/cpp/src/umap/runner.cuh:136 Done. Calling fuzzy simplicial set
[D] [12:58:44.422284] /__w/cuml/cuml/cpp/src/umap/fuzzy_simpl_set/naive.cuh:317 Smooth kNN Distances
[D] [12:58:44.423164] /__w/cuml/cuml/cpp/src/umap/fuzzy_simpl_set/naive.cuh:319 sigmas = [ 0.0667543, 0.294415, 0.109013, 0.320028, 0.269716, 0.421898, 0.0762148, 0.38213, 0.294661, 0.165405, 0.218035, 0.0997238, 0.146852, 0.289604, 0.35952, 0.148159, 0.126254, 0.152779, 0.126077, 0.311195, 0.253, 0.38686, 0.0667157, 0.161199, 0.162477 ]

[D] [12:58:44.423207] /__w/cuml/cuml/cpp/src/umap/fuzzy_simpl_set/naive.cuh:321 rhos = [ 6.40256, 7.4475, 7.96912, 5.50156, 6

Lastly, run Leiden clustering on samples

In [ ]:
loop_generator = itertools.product(possible_tiers, possible_subsets, possible_log, possible_explained_var, possible_cluster_k)

for selected_tier, selected_subset, use_log, explained_var, n_neighbors in loop_generator:
    dataset_prefix = "adata_" + selected_tier + "_SUBSET_" + selected_subset + "_LAYER_" + selected_layer + "_LOG_" + str(use_log)
    file_prefix = dataset_prefix + "_EXPLAINEDVAR_%d_KNN_%d" % (int(explained_var * 100), n_neighbors)
    output_file = os.path.join(obs_output_path, file_prefix + "__obs.h5ad")
    if os.path.exists(output_file):
        continue
    adata_file = os.path.join(adata_base_output_path, dataset_prefix + ".h5ad")
    knn_file = os.path.join(knn_output_path, file_prefix + "__knn.h5ad")
    print("Reading " + knn_file)
    adata_knn = sc.read_h5ad(knn_file)
    print("Reading " + adata_file)
    adata_base = sc.read_h5ad(adata_file)
    print("Computing Leiden clusters")
    G, edges = get_cugraph_from_array(adata_knn.obsm['indices'])
    clusters = leiden_gpu(G)
    adata = sc.AnnData(scipy.sparse.csr_matrix([[0] * n_neighbors] * adata_knn.n_obs))
    adata.obs_names = adata_knn.obs_names
    adata.obs['Leiden_cluster'] = clusters.copy()
    sc.write(output_file, adata = adata)
